# Metric Glance Span-Detection Classifier

Pipeline: export from D1 -> explore labels -> prepare features -> train -> export to ONNX.

The trained model will slot into `proposeSpans()` in `extension/converter.js` behind the
`useEncoder` flag. The arithmetic (conversion factors) is never touched by the model.

## Setup

```bash
# from the train/ directory
uv sync
uv run jupyter notebook classifier.ipynb
```

In [ ]:
# Print the path to the Python interpreter
!where python

In [ ]:
# All imports for the notebook live here; run this cell first. Imports for the
# not-yet-written train and ONNX-export sections will be added when those land.
import json
import os
import pathlib
import subprocess
import sys
import warnings
from typing import cast, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split

In [ ]:
# Print environment information


# Function to print environment information
def print_directory_info(target: str | None = None) -> None:
    """
    Prints the current working directory, active virtual environment, and Python version.
    If target is provided, navigates to that directory first.
    """
    # This is done in a function mainly to avoid cluttering the global namespace


    # Helper to make sure the working directory ends in a specific folder
    def _ensure_dir(target_path: str, current_path: str = os.getcwd()) -> str:
        """
        Recursively walks up the directory tree to find a folder matching target_path.
        Returns the full absolute path of the matched folder, or raises FileNotFoundError.
        """

        # If the current path is the target path, return the current path
        if os.path.basename(current_path) == target_path:
            return current_path

        # else, if the current path is the top level directory, return an error
        elif os.path.dirname(current_path) == current_path:
            raise FileNotFoundError(f"Could not find directory {target_path}")

        # In all other cases, call the function again with the parent directory
        else:
            return _ensure_dir(target_path, os.path.dirname(current_path))


    # Add warning if if target has spaces
    if target and " " in target:
        warnings.warn(
        "The target directory name contains spaces. GitHub repository names cannot include spaces. Spaces are typically replaced with hyphens (-) in the URL. This may cause confusion when pushing, pulling, or cloning. The current working directory is: \n" + os.getcwd()
        )

    # if a target is specified, change the current working directory to that folder
    if target:
        os.chdir(_ensure_dir(target))

    # Print a separator line
    print("-" * 100)

    # Print the current working directory
    current_directory: str = os.getcwd()
    parent_directory: str = os.path.dirname(current_directory)
    grandparent_directory: str = os.path.dirname(parent_directory)

    print(
        "Current working directory: "
        + f"{os.path.basename(grandparent_directory)}\\{os.path.basename(parent_directory)}\\{os.path.basename(current_directory)}"
    )

    # Check if a virtual environment is active
    venv_path: str | None = os.getenv("VIRTUAL_ENV")
    if venv_path:
        venv_name: str = os.path.basename(venv_path)
        _, dir_name = os.path.split(os.path.split(venv_path)[0])
        print(f"Environment: {dir_name}/\033[1m{venv_name}\033[0m")  # bold
    else:
        print("\033[1m\033[3m" + "No virtual environment" + "\033[0m")  # bold + italic

    # Print the Python version
    print("Python version: " + sys.version)

    # Print another separator line
    print("-" * 100)



# Call the function
print_directory_info("metric-glance")
# Replace "expected current directory name" with the actual name of the directory you expect to be in.

In [ ]:
# Resolve paths regardless of whether CWD is train/ or the repo root
# (an os.chdir to the repo root in another cell would break relative paths).
_cwd: pathlib.Path = pathlib.Path.cwd()
collect_dir: pathlib.Path
data_dir: pathlib.Path
if (_cwd / "collect").exists():      # CWD is already the repo root
    collect_dir = (_cwd / "collect").resolve()
    data_dir    = (_cwd / "train" / "data").resolve()
else:                                 # CWD is train/
    collect_dir = (_cwd / ".." / "collect").resolve()
    data_dir    = (_cwd / "data").resolve()

data_dir.mkdir(parents=True, exist_ok=True)

print(f"collect_dir: {collect_dir}")
print(f"data_dir:    {data_dir}")

---
## 1. Authenticate with Cloudflare and export from D1

`wrangler` needs an active Cloudflare session to hit the remote D1 database.
Run the cell below first. If it prints your email, you are ready. If it errors,
open a terminal in `../collect/` and run:

```bash
npx wrangler login
```

That opens a browser OAuth flow. Come back and re-run the cell once it finishes.

Alternatively, set the `CLOUDFLARE_API_TOKEN` environment variable to a token
that has D1 read permissions, and wrangler will use that instead of the browser flow.

In [ ]:
# Check wrangler authentication before doing anything else. If collect_dir did
# not resolve to a real directory, fall back to searching parent dirs; the
# corrected path lives in collect_cwd (collect_dir itself is left as-is).
collect_cwd: pathlib.Path = pathlib.Path(collect_dir)

if not collect_cwd.is_dir():
    found: pathlib.Path | None = None
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        candidate: pathlib.Path = base / "collect"
        if candidate.is_dir():
            found = candidate.resolve()
            break
    if found is None:
        raise FileNotFoundError(
            f"Could not find a valid collect/ directory. Current collect_dir: {collect_dir}"
        )
    collect_cwd = found
else:
    collect_cwd = collect_cwd.resolve()

_flags: int = subprocess.CREATE_NO_WINDOW if sys.platform == "win32" else 0

result: subprocess.CompletedProcess[str] = subprocess.run(
    "npx wrangler whoami",
    capture_output=True,
    text=True,
    encoding="utf-8",
    stdin=subprocess.DEVNULL,
    cwd=str(collect_cwd),
    shell=True,
    creationflags=_flags,
)
stdout: str = result.stdout or ""
stderr: str = result.stderr or ""
combined: str = f"{stdout}\n{stderr}".lower()

if result.returncode != 0 or "not authenticated" in combined:
    raise RuntimeError(
        "wrangler is not authenticated.\n"
        "Run this in a terminal from the collect/ directory:\n"
        "    npx wrangler login\n"
        "Then re-run this cell.\n\n"
        f"stdout:\n{stdout}\n\nstderr:\n{stderr}"
    )
if stdout.strip():
    print(stdout.strip())
elif stderr.strip():
    print(stderr.strip())
else:
    print("wrangler is authenticated.")

In [ ]:
# Pull fresh data from D1 on every run, then load it.
#
# The actual wrangler/Node call lives in train/refresh_data.py and runs as a
# SEPARATE process. The kernel only ever spawns python.exe here, never Node,
# so the Jupyter-on-Windows Node crash (0xC0000409) can at worst make this
# subprocess fail; it can never take down the kernel. On any failure we fall
# back to the last cached submissions.json.
#
# Set REFRESH_ON_RUN = False to skip the D1 pull and just use the cached file
# (offline, or to avoid the ~few-second wrangler round-trip).
REFRESH_ON_RUN: bool = True

EXPORT_FILE: pathlib.Path = data_dir / "submissions.json"
REFRESH_SCRIPT: pathlib.Path = (
    data_dir.parent / "refresh_data.py"
)  # train/refresh_data.py


def _refresh_from_d1() -> tuple[bool, str]:
    """Run refresh_data.py in a separate process. Returns (ok, message)."""
    flags: int = subprocess.CREATE_NO_WINDOW if sys.platform == "win32" else 0
    try:
        p: subprocess.CompletedProcess[str] = subprocess.run(
            [sys.executable, str(REFRESH_SCRIPT)],
            capture_output=True,
            text=True,
            encoding="utf-8",
            stdin=subprocess.DEVNULL,
            creationflags=flags,
            timeout=180,
        )
    except Exception as e:  # never let a refresh problem abort the notebook
        return False, f"could not start refresh_data.py: {e}"
    if p.returncode != 0:
        return False, (p.stderr or p.stdout or f"exit {p.returncode}").strip()
    return True, (p.stdout or "").strip()


if REFRESH_ON_RUN:
    ok, msg = _refresh_from_d1()
    print(
        f"D1 refresh: {msg}"
        if ok
        else f"WARNING: refresh failed, using cached file.\n{msg}"
    )

if not EXPORT_FILE.exists():
    raise FileNotFoundError(
        f"No data at {EXPORT_FILE} and the D1 refresh did not produce one.\n"
        "Run this from a terminal in the train/ directory to diagnose:\n"
        "    python refresh_data.py"
    )

rows: list[dict[str, object]] = json.loads(EXPORT_FILE.read_text(encoding="utf-8"))

df_raw: pd.DataFrame = pd.DataFrame(rows)

print(f"Loaded {len(df_raw):,} rows from {EXPORT_FILE.name}  (shape {df_raw.shape})")

df_raw.head()

---
## 2. Trusted installs (sample weighting)

`install_id` groups rows by who submitted them (a random per-install id, not PII).
Corrections from trusted installs (e.g. your own) should count for more during
training. The trusted id(s) are read from a local, gitignored `.env` file, never
hardcoded, so they stay out of this open-source repo. See `.env.example` for how
to find yours.

In [ ]:
# Load train/.env (gitignored). The real id(s) live there, not in this notebook.
load_dotenv(data_dir.parent / ".env")

TRUSTED_INSTALL_IDS: set[str] = {
    s.strip() for s in os.getenv("MG_TRUSTED_INSTALL_IDS", "").split(",") if s.strip()
}

# Per-row weight: trusted installs count most, then corrected, then seen/auto.
TIER_WEIGHT: dict[str, float] = {"corrected": 3.0, "seen": 1.0, "auto": 0.5}


def row_weight(r: pd.Series[Any]) -> float:
    install_id: str = str(r.get("install_id", ""))
    tier: str = str(r.get("tier", ""))

    if TRUSTED_INSTALL_IDS and install_id in TRUSTED_INSTALL_IDS:
        return 5.0
    return TIER_WEIGHT.get(tier, 1.0)


if "install_id" not in df_raw.columns:
    print(
        "NOTE: no install_id column yet. Re-run the data-load cell to pull it from D1."
    )
else:
    df_raw["sample_weight"] = df_raw.apply(row_weight, axis=1)
    n_trusted: int = (
        int(df_raw["install_id"].isin(TRUSTED_INSTALL_IDS).sum())
        if TRUSTED_INSTALL_IDS
        else 0
    )
    # Print only aggregates, never the ids themselves (this output may be committed).
    print(f"Distinct installs:      {df_raw['install_id'].nunique():,}")
    print(f"Trusted ids configured: {len(TRUSTED_INSTALL_IDS)}")
    print(f"Rows from trusted:      {n_trusted:,}")
    print("\nWeight distribution:")
    print(df_raw["sample_weight"].value_counts().sort_index())

---
## 3. Explore the labels

Before modeling, see what is actually in the data: which tiers and labels
dominate, how the target maps from `label` and `unit_id`, where the negatives
are, and how concentrated the data is across installs. Everything printed here
is aggregate (counts), never raw rows, so the output is safe to commit.

In [ ]:
# Aggregate-only exploration (safe to commit: no raw rows, no install ids printed).
print(f"=== {len(df_raw):,} rows, {df_raw.shape[1]} columns ===")

tier_counts: pd.Series = df_raw["tier"].value_counts(dropna=False)
print("\n--- tier (how the label was obtained) ---")
print(tier_counts.to_string())

prefix_counts: pd.Series = df_raw["label"].str.split(":").str[0].value_counts(dropna=False)
print("\n--- label prefix (the action) ---")
print(prefix_counts.to_string())

label_counts: pd.Series = df_raw["label"].value_counts(dropna=False)
print("\n--- top labels ---")
print(label_counts.head(15).to_string())
print(f"\ndistinct labels: {df_raw['label'].nunique()}")

unit_counts: pd.Series = df_raw["unit_id"].replace("", pd.NA).value_counts(dropna=False)
print("\n--- top unit_id (the conversion target) ---")
print(unit_counts.head(12).to_string())

# Blank counts per column: confirms which blanks are structural (boundary spans,
# missing headings) versus columns that are mostly empty.
blank: pd.Series = df_raw.astype(str).apply(lambda s: (s.str.strip() == "").sum())
print("\n--- blank-string counts per column (nonzero) ---")
print(blank[blank > 0].sort_values(ascending=False).to_string())

# How concentrated is the data across installs? (counts only, ids not shown)
per_install: pd.Series = df_raw.groupby("install_id").size().sort_values(ascending=False)
print(f"\n--- {df_raw['install_id'].nunique()} distinct installs, rows each ---")
print(per_install.rename("rows").reset_index(drop=True).to_string())

n_neg: int = int(df_raw["label"].str.startswith("not_a_conversion").sum())
print(f"\nnegative examples (not_a_conversion): {n_neg}")

# Visual: the label distribution at a glance.
ax: Axes = df_raw["label"].value_counts().head(12).iloc[::-1].plot.barh(figsize=(7, 4))
ax.set_title("Top 12 labels")
ax.set_xlabel("rows")
plt.tight_layout()
plt.show()

---
## 4. Prepare features

Build the modeling frame for a baseline **sequence classifier**: given a candidate
span and its surrounding text, predict the unit (or `__none__` for a false
positive). The plan is a `TfidfVectorizer` + `LogisticRegression` pipeline (its
imports go in the train section, when written); the BERT token-classifier in the
roadmap is a later step.

Decisions, all adjustable at the top of the cell:

- **Target `y`**: `not_a_conversion` becomes `__none__` (the negative class);
  otherwise the `unit_id` (reliable, it matches the label suffix ~99.6% of the
  time). Generic `seen:unit` hovers have no specific unit and are dropped.
- **Text feature**: `before_ctx [ span ] after_ctx`, so the model sees the surface
  form and the disambiguating context together, with the span bracketed.
- **Rare classes**: classes with fewer than `MIN_PER_CLASS` examples are dropped so
  the split can be stratified. What is dropped is printed, not silent.
- **Split**: stratified 80/20. A group-by-`install_id` split would avoid per-install
  leakage, but with only a few installs contributing today it is degenerate (most
  classes land entirely on one side). Revisit once more installs contribute.

In [ ]:
NONE_CLASS: str = "__none__"   # negative class (a flagged false positive)
MIN_PER_CLASS: int = 2         # drop classes rarer than this so the split can stratify
TEST_SIZE: float = 0.20
RANDOM_STATE: int = 42


def _clean(s: pd.Series) -> pd.Series:
    """Collapse runs of whitespace and strip; treat NaN as empty string."""
    return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()


# --- target y: not_a_conversion -> negative class; otherwise the unit_id ---
prefix: pd.Series = df_raw["label"].str.split(":").str[0]
unit: pd.Series = _clean(df_raw["unit_id"]).replace("", np.nan)
y_values: np.ndarray = np.where(prefix.eq("not_a_conversion"), NONE_CLASS, unit)
df_raw["y"] = y_values

# --- text feature: surrounding context with the span bracketed ---
# The model sees the surface form (which often determines the unit) plus the
# context that disambiguates the ambiguous cases (US vs imperial, etc.).
text_feature: pd.Series = (
    _clean(df_raw["before_ctx"]) + " [ " + _clean(df_raw["span"]) + " ] " + _clean(df_raw["after_ctx"])
).str.strip()
df_raw["text"] = text_feature

# --- drop rows we cannot label (generic seen:unit hovers have no unit_id) ---
usable: pd.DataFrame = df_raw[df_raw["y"].notna()].copy()
print(f"usable rows: {len(usable)} of {len(df_raw)}  (dropped {len(df_raw) - len(usable)} with no unit label)")

# --- drop rare classes so train/test can stratify; report, never silent ---
counts: pd.Series = usable["y"].value_counts()
rare: pd.Series = counts[counts < MIN_PER_CLASS]
if len(rare):
    print(f"dropping {len(rare)} classes with < {MIN_PER_CLASS} examples "
          f"({int(rare.sum())} rows): {list(rare.index)}")
data: pd.DataFrame = usable[usable["y"].isin(counts[counts >= MIN_PER_CLASS].index)].copy()

# --- sample_weight from section 2 (fall back to tier weights if absent) ---
if "sample_weight" not in data.columns:
    data["sample_weight"] = data["tier"].map(TIER_WEIGHT).fillna(1.0)

# --- stratified 80/20 split (see note above re: group-by-install) ---
# train_test_split is typed to return list[Any]; cast so the unpacked frames
# carry known types.
train_df, test_df = cast(
    "tuple[pd.DataFrame, pd.DataFrame]",
    train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["y"]),
)

X_train: pd.Series = train_df["text"]
y_train: pd.Series = train_df["y"]
w_train: pd.Series = train_df["sample_weight"]
X_test: pd.Series = test_df["text"]
y_test: pd.Series = test_df["y"]
w_test: pd.Series = test_df["sample_weight"]

print(f"\ntrain {len(train_df)}   test {len(test_df)}")
print(f"classes: {data['y'].nunique()}  (train {y_train.nunique()}, test {y_test.nunique()})")
print("\ntarget balance (top):")
print(data["y"].value_counts().head(8).to_string())